# Genomic Sequence Analysis: GC Content, K-mer Profiling & Repeat Detection

**Author:** Samuel Ukwueze  
**Date:** March 2026  

---

This notebook walks through some foundational sequence-level analysis I find myself doing fairly often when working with new genomic data. The goal here is to characterize sequences before any alignment or annotation steps like GC content distribution, k-mer frequency profiles, and basic repeat detection. These are the kinds of checks that tell you a lot about your data before you commit to a heavier pipeline.

I'm using synthetic FASTA sequences here, but the logic transfers directly to real genomic data.

## 1. Imports and Configuration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from collections import Counter
import re
import warnings
warnings.filterwarnings('ignore')

np.random.seed(7)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'sans-serif'

print('Ready.')

## 2. Generating Synthetic FASTA Sequences

I'm generating 20 sequences of varying length with different GC biases to simulate what you'd see across different genomic regions- some AT-rich intergenic regions, some GC-rich coding sequences. In real work you'd just parse an actual FASTA file with Biopython, but this keeps the notebook self-contained.

In [ ]:
def generate_sequence(length, gc_content):
    """Generate a random DNA sequence with a specified GC content."""
    at_content = 1 - gc_content
    bases = np.random.choice(
        ['A', 'T', 'G', 'C'],
        size=length,
        p=[at_content/2, at_content/2, gc_content/2, gc_content/2]
    )
    return ''.join(bases)

# Mix of AT-rich and GC-rich sequences
gc_profiles = np.random.uniform(0.30, 0.72, 20)
lengths = np.random.randint(800, 3000, 20)

sequences = {}
for i, (length, gc) in enumerate(zip(lengths, gc_profiles)):
    seq_id = f'Seq_{i+1:02d}'
    sequences[seq_id] = generate_sequence(length, gc)

print(f'Generated {len(sequences)} sequences')
print(f'Length range: {min(lengths)} — {max(lengths)} bp')

# Preview
for k, v in list(sequences.items())[:3]:
    print(f'{k} | Length: {len(v)} | First 60bp: {v[:60]}')

## 3. GC Content Analysis

GC content is one of the most basic but informative sequence properties. It affects things like melting temperature, secondary structure stability, and even codon usage. I'm calculating it both globally per sequence and in sliding windows to catch local variation within sequences.

In [ ]:
def gc_content(seq):
    seq = seq.upper()
    gc = seq.count('G') + seq.count('C')
    return gc / len(seq) if len(seq) > 0 else 0

def sliding_window_gc(seq, window=100, step=50):
    positions, gc_vals = [], []
    for i in range(0, len(seq) - window, step):
        positions.append(i + window // 2)
        gc_vals.append(gc_content(seq[i:i+window]))
    return positions, gc_vals

gc_data = pd.DataFrame({
    'sequence': list(sequences.keys()),
    'length': [len(v) for v in sequences.values()],
    'gc_content': [gc_content(v) for v in sequences.values()]
})

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# GC distribution
axes[0].hist(gc_data['gc_content'], bins=12, color='#4C72B0', edgecolor='white', linewidth=0.8)
axes[0].axvline(gc_data['gc_content'].mean(), color='#E74C3C', linestyle='--',
                linewidth=1.5, label=f"Mean: {gc_data['gc_content'].mean():.2f}")
axes[0].set_xlabel('GC Content')
axes[0].set_ylabel('Number of Sequences')
axes[0].set_title('GC Content Distribution Across Sequences')
axes[0].legend()

# GC vs length scatter
axes[1].scatter(gc_data['length'], gc_data['gc_content'],
                color='#2ECC71', edgecolors='white', s=70, linewidth=0.8)
for _, row in gc_data.iterrows():
    axes[1].annotate(row['sequence'], (row['length'], row['gc_content']),
                     fontsize=6, xytext=(3, 2), textcoords='offset points')
axes[1].set_xlabel('Sequence Length (bp)')
axes[1].set_ylabel('GC Content')
axes[1].set_title('GC Content vs Sequence Length')

plt.tight_layout()
plt.show()

print(gc_data.sort_values('gc_content', ascending=False).to_string(index=False))

## 4. Sliding Window GC — Intra-sequence Variation

Global GC can hide a lot of local structure. I'll pick a few sequences and look at how GC fluctuates along the length. This kind of profile is useful for spotting CpG islands, repeat regions, or structural features.

In [ ]:
selected = list(sequences.keys())[:4]

fig, axes = plt.subplots(2, 2, figsize=(12, 7))
axes = axes.flatten()

colors = ['#3498DB', '#E74C3C', '#2ECC71', '#9B59B6']

for idx, (seq_id, color) in enumerate(zip(selected, colors)):
    seq = sequences[seq_id]
    positions, gc_vals = sliding_window_gc(seq, window=100, step=25)
    global_gc = gc_content(seq)

    axes[idx].plot(positions, gc_vals, color=color, linewidth=1.2, alpha=0.9)
    axes[idx].axhline(global_gc, color='black', linestyle='--', linewidth=0.8,
                      label=f'Global GC: {global_gc:.2f}')
    axes[idx].fill_between(positions, gc_vals, global_gc,
                           where=[g > global_gc for g in gc_vals],
                           alpha=0.2, color='red', label='Above mean')
    axes[idx].fill_between(positions, gc_vals, global_gc,
                           where=[g <= global_gc for g in gc_vals],
                           alpha=0.2, color='blue', label='Below mean')
    axes[idx].set_title(f'{seq_id} | Length: {len(seq)} bp')
    axes[idx].set_xlabel('Position (bp)')
    axes[idx].set_ylabel('GC Content (100bp window)')
    axes[idx].legend(fontsize=7)
    axes[idx].set_ylim(0, 1)

plt.suptitle('Sliding Window GC Content (Window=100bp, Step=25bp)', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

## 5. K-mer Frequency Profiling

K-mer profiles are surprisingly powerful: they capture sequence composition in a way that doesn't require alignment. I'm looking at 3-mers (trinucleotides) here since they're biologically meaningful (codon structure) and computationally manageable.

In [ ]:
def kmer_counts(seq, k=3):
    kmers = [seq[i:i+k] for i in range(len(seq) - k + 1)]
    return Counter(kmers)

def kmer_frequencies(seq, k=3):
    counts = kmer_counts(seq, k)
    total = sum(counts.values())
    return {kmer: count / total for kmer, count in counts.items()}

k = 3
all_kmers = set()
freq_profiles = {}

for seq_id, seq in sequences.items():
    freq = kmer_frequencies(seq, k)
    freq_profiles[seq_id] = freq
    all_kmers.update(freq.keys())

kmer_df = pd.DataFrame(freq_profiles).T.fillna(0)
kmer_df = kmer_df[sorted(kmer_df.columns)]

print(f'Total unique {k}-mers observed: {len(all_kmers)}')
print(f'Expected possible {k}-mers: {4**k}')

# Top 20 most variable k-mers
top_variable = kmer_df.var().nlargest(20).index

fig, ax = plt.subplots(figsize=(14, 5))
kmer_df[top_variable].T.plot(kind='bar', ax=ax, legend=False, colormap='tab20', width=0.8)
ax.set_title(f'Top 20 Most Variable {k}-mers Across Sequences')
ax.set_xlabel(f'{k}-mer')
ax.set_ylabel('Relative Frequency')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

## 6. K-mer Heatmap: Sequence Similarity

By treating each sequence as a vector of k-mer frequencies, I can compute pairwise similarity between sequences. This is essentially alignment-free sequence comparison — useful when you're dealing with highly diverged sequences or large datasets where alignment is too slow.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

kmer_matrix = kmer_df.values
similarity_matrix = cosine_similarity(kmer_matrix)
sim_df = pd.DataFrame(similarity_matrix,
                       index=kmer_df.index,
                       columns=kmer_df.index)

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.eye(len(sim_df), dtype=bool)  # mask diagonal
sns.heatmap(sim_df, cmap='YlOrRd', annot=False, fmt='.2f',
            linewidths=0.3, linecolor='white', ax=ax,
            cbar_kws={'label': 'Cosine Similarity'},
            vmin=0.8, vmax=1.0)
ax.set_title(f'Pairwise Sequence Similarity Based on {k}-mer Profiles', fontsize=12)
ax.set_xlabel('Sequence')
ax.set_ylabel('Sequence')
plt.tight_layout()
plt.show()

# Most similar pair (excluding self)
np.fill_diagonal(similarity_matrix, 0)
max_idx = np.unravel_index(np.argmax(similarity_matrix), similarity_matrix.shape)
seq_names = list(kmer_df.index)
print(f'Most similar pair: {seq_names[max_idx[0]]} & {seq_names[max_idx[1]]} '
      f'(similarity: {similarity_matrix[max_idx]:.4f})')

## 7. Simple Repeat Detection

Repetitive elements are everywhere in genomes and they can cause real headaches like assembly errors, alignment artifacts, inflated read counts. This is a basic regex-based scan for simple tandem repeats (STRs), so  I will be looking for motifs of length 2–4 repeated at least 3 times.

In [ ]:
def find_simple_repeats(seq, min_motif=2, max_motif=4, min_repeats=3):
    repeats = []
    for motif_len in range(min_motif, max_motif + 1):
        pattern = r'(([ACGT]{' + str(motif_len) + r'})\2{' + str(min_repeats - 1) + r',})'
        for match in re.finditer(pattern, seq):
            repeats.append({
                'start': match.start(),
                'end': match.end(),
                'length': match.end() - match.start(),
                'motif': match.group(2),
                'motif_len': motif_len,
                'repeat_count': len(match.group(1)) // motif_len
            })
    return repeats

repeat_summary = []
for seq_id, seq in sequences.items():
    repeats = find_simple_repeats(seq)
    total_repeat_bp = sum(r['length'] for r in repeats)
    repeat_summary.append({
        'sequence': seq_id,
        'n_repeats': len(repeats),
        'total_repeat_bp': total_repeat_bp,
        'repeat_fraction': total_repeat_bp / len(seq),
        'seq_length': len(seq)
    })

repeat_df = pd.DataFrame(repeat_summary).sort_values('repeat_fraction', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].barh(repeat_df['sequence'], repeat_df['n_repeats'],
             color='#9B59B6', edgecolor='white', linewidth=0.5)
axes[0].set_xlabel('Number of Simple Repeats Detected')
axes[0].set_title('Simple Repeat Count per Sequence')

axes[1].barh(repeat_df['sequence'], repeat_df['repeat_fraction'] * 100,
             color='#E67E22', edgecolor='white', linewidth=0.5)
axes[1].set_xlabel('Repeat Content (%)')
axes[1].set_title('Percentage of Sequence Covered by Repeats')

plt.tight_layout()
plt.show()

print(repeat_df.to_string(index=False))

## 8. Nucleotide Composition Summary

So here is a quick per-sequence breakdown of all four nucleotides. Useful for flagging anything unusual like sequences with unexpectedly high N content in real data, or weird base biases that might suggest contamination.

In [ ]:
composition = []
for seq_id, seq in sequences.items():
    total = len(seq)
    composition.append({
        'sequence': seq_id,
        'A': seq.count('A') / total,
        'T': seq.count('T') / total,
        'G': seq.count('G') / total,
        'C': seq.count('C') / total,
    })

comp_df = pd.DataFrame(composition).set_index('sequence')

fig, ax = plt.subplots(figsize=(12, 5))
comp_df.plot(kind='bar', stacked=True, ax=ax,
             color=['#3498DB', '#E74C3C', '#2ECC71', '#F39C12'],
             edgecolor='white', linewidth=0.4, width=0.8)
ax.set_xlabel('Sequence')
ax.set_ylabel('Nucleotide Fraction')
ax.set_title('Nucleotide Composition per Sequence')
ax.tick_params(axis='x', rotation=45)
ax.legend(title='Base', bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 9. Takeaways

This kind of sequence-level characterization doesn't get talked about much, but I think it's worth doing before jumping straight into alignment. You pick up on things like unusual GC distributions, high repeat content, compositional outliers that would otherwise only show up as weird artifacts downstream.

For real genomic data, I'd swap the synthetic sequences for actual FASTA input via Biopython's `SeqIO`, and I'd probably add a BLAST step after the k-mer similarity analysis to confirm whether the sequences that cluster together share any known biological relationship.

---
*Analysis performed in Python 3.12 | Libraries: NumPy, Pandas, Matplotlib, Seaborn, Scikit-learn, Collections, Re*